In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn tqdm seaborn

In [ ]:
import os
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet50, efficientnet_b0

warnings.filterwarnings("ignore")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device : {device}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

In [ ]:
import os

DATASET_PATH = "/kaggle/input/datasets/msambare/fer2013"

ORIGINAL_TRAIN = os.path.join(DATASET_PATH, "train")

NEW_TRAIN = "/kaggle/working/train"
NEW_TEST = "/kaggle/working/test"

MODEL_DIR = "/kaggle/working/models"

os.makedirs(NEW_TRAIN, exist_ok=True)
os.makedirs(NEW_TEST, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
classes = sorted(os.listdir(ORIGINAL_TRAIN))

print(classes)
print("Number of classes:", len(classes))

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

split_ratio = 0.7

for cls in classes:

    src = os.path.join(ORIGINAL_TRAIN, cls)

    dst_train = os.path.join(NEW_TRAIN, cls)
    dst_test = os.path.join(NEW_TEST, cls)

    os.makedirs(dst_train, exist_ok=True)
    os.makedirs(dst_test, exist_ok=True)

    images = [
        img for img in os.listdir(src)
        if img.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    train_imgs, test_imgs = train_test_split(
        images,
        train_size=split_ratio,
        random_state=42,
        shuffle=True
    )

    for img in train_imgs:
        shutil.copy(
            os.path.join(src, img),
            os.path.join(dst_train, img)
        )

    for img in test_imgs:
        shutil.copy(
            os.path.join(src, img),
            os.path.join(dst_test, img)
        )

print("70/30 split completed.")

In [ ]:
for cls in classes:

    train_count = len(os.listdir(os.path.join(NEW_TRAIN, cls)))
    test_count = len(os.listdir(os.path.join(NEW_TEST, cls)))

    total = train_count + test_count

    print(f"{cls:10s} Train: {train_count:5d} Test: {test_count:5d} Total: {total}")

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1,0.1),
        scale=(0.9,1.1)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

In [ ]:
train_dataset = ImageFolder(
    NEW_TRAIN,
    transform=train_transform
)

test_dataset = ImageFolder(
    NEW_TEST,
    transform=test_transform
)

print("Training Images :", len(train_dataset))
print("Testing Images :", len(test_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

In [ ]:
class_names = train_dataset.classes

print(class_names)
print("Number of classes:",len(class_names))

In [ ]:
images,labels = next(iter(train_loader))

plt.figure(figsize=(12,8))

for i in range(12):

    plt.subplot(3,4,i+1)

    img = images[i].permute(1,2,0).numpy()

    img = img*0.5+0.5

    plt.imshow(img)

    plt.title(class_names[labels[i]])

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
for cls in class_names:

    count=len(os.listdir(os.path.join(NEW_TRAIN,cls)))

    print(f"{cls:10s}: {count}")

In [ ]:
EPOCHS = 30

LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader):

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)

    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
def validate(model, loader, criterion):

    model.eval()

    running_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    loss = running_loss / len(loader)

    acc = 100 * correct / total

    return loss, acc

In [ ]:
class CustomCNN(nn.Module):

    def __init__(self):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 7)

        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x

In [ ]:
customcnn = CustomCNN().to(device)

optimizer = optim.Adam(
    customcnn.parameters(),
    lr=LEARNING_RATE
)

print(customcnn)

In [ ]:
train_losses = []
train_accs = []

val_losses = []
val_accs = []

for epoch in range(EPOCHS):

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")

    train_loss, train_acc = train_one_epoch(
        customcnn,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = validate(
        customcnn,
        test_loader,
        criterion
    )

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Accuracy : {val_acc:.2f}%")

print("Training Finished!")

In [ ]:
# Save only after the entire training is complete
torch.save(customcnn.state_dict(), "/kaggle/working/models/customcnn.pth")

print("Custom CNN model saved successfully!")

In [ ]:
from torchvision.models import resnet50

resnet_model = resnet50(weights=None)

resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    7
)

resnet_model = resnet_model.to(device)

print(resnet_model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    resnet_model.parameters(),
    lr=LEARNING_RATE
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.1
)

In [ ]:
resnet_train_loss = []
resnet_train_acc = []

resnet_val_loss = []
resnet_val_acc = []

In [ ]:
for epoch in range(EPOCHS):

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")

    train_loss, train_acc = train_one_epoch(
        resnet_model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = validate(
        resnet_model,
        test_loader,
        criterion
    )

    scheduler.step()

    resnet_train_loss.append(train_loss)
    resnet_train_acc.append(train_acc)

    resnet_val_loss.append(val_loss)
    resnet_val_acc.append(val_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Accuracy : {val_acc:.2f}%")

print("\nResNet50 Training Completed!")

In [ ]:
torch.save(
    resnet_model.state_dict(),
    os.path.join(MODEL_DIR, "resnet50.pth")
)

print("ResNet50 model saved successfully!")

In [ ]:
from torchvision.models import efficientnet_b0

efficientnet_model = efficientnet_b0(weights=None)

efficientnet_model.classifier[1] = nn.Linear(
    efficientnet_model.classifier[1].in_features,
    7
)

efficientnet_model = efficientnet_model.to(device)

print(efficientnet_model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    efficientnet_model.parameters(),
    lr=LEARNING_RATE
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.1
)

In [ ]:
efficientnet_train_loss = []
efficientnet_train_acc = []

efficientnet_val_loss = []
efficientnet_val_acc = []

In [ ]:
for epoch in range(EPOCHS):

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")

    train_loss, train_acc = train_one_epoch(
        efficientnet_model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = validate(
        efficientnet_model,
        test_loader,
        criterion
    )

    scheduler.step()

    efficientnet_train_loss.append(train_loss)
    efficientnet_train_acc.append(train_acc)

    efficientnet_val_loss.append(val_loss)
    efficientnet_val_acc.append(val_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Accuracy : {val_acc:.2f}%")

print("\nEfficientNet Training Completed!")

In [ ]:
torch.save(
    efficientnet_model.state_dict(),
    os.path.join(MODEL_DIR, "efficientnet.pth")
)

print("EfficientNet model saved successfully!")

In [ ]:
# Load Custom CNN
customcnn = CustomCNN().to(device)
customcnn.load_state_dict(torch.load(os.path.join(MODEL_DIR, "customcnn.pth"), map_location=device))
customcnn.eval()

# Load ResNet50
resnet_model = resnet50(weights=None)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 7)
resnet_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "resnet50.pth"), map_location=device))
resnet_model = resnet_model.to(device)
resnet_model.eval()

# Load EfficientNet
efficientnet_model = efficientnet_b0(weights=None)
efficientnet_model.classifier[1] = nn.Linear(
    efficientnet_model.classifier[1].in_features,
    7
)
efficientnet_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "efficientnet.pth"), map_location=device))
efficientnet_model = efficientnet_model.to(device)
efficientnet_model.eval()

print("All models loaded successfully.")

In [ ]:
import torch.nn.functional as F

def predict_model(model, loader):

    y_true = []
    y_pred = []
    y_prob = []

    model.eval()

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device)

            outputs = model(images)

            probs = F.softmax(outputs, dim=1)

            preds = torch.argmax(probs, dim=1)

            y_true.extend(labels.numpy())

            y_pred.extend(preds.cpu().numpy())

            y_prob.extend(probs.cpu().numpy())

    return (
        np.array(y_true),
        np.array(y_pred),
        np.array(y_prob)
    )

In [ ]:
true_labels, cnn_pred, cnn_prob = predict_model(customcnn, test_loader)

_, resnet_pred, resnet_prob = predict_model(resnet_model, test_loader)

_, efficient_pred, efficient_prob = predict_model(efficientnet_model, test_loader)

In [ ]:
def evaluate_model(name, y_true, y_pred):

    print("="*50)

    print(name)

    print("="*50)

    print("Accuracy :",
          accuracy_score(y_true,y_pred))

    print("Precision:",
          precision_score(
              y_true,
              y_pred,
              average="weighted"
          ))

    print("Recall:",
          recall_score(
              y_true,
              y_pred,
              average="weighted"
          ))

    print("F1 Score:",
          f1_score(
              y_true,
              y_pred,
              average="weighted"
          ))

    print()

    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names
    ))

In [ ]:
evaluate_model(
    "DST Fusion",
    true_labels,
    dst_pred
)

plot_confusion(
    true_labels,
    dst_pred,
    "DST Fusion"
)

In [ ]:
evaluate_model(
    "Custom CNN",
    true_labels,
    cnn_pred
)

evaluate_model(
    "ResNet50",
    true_labels,
    resnet_pred
)

evaluate_model(
    "EfficientNet",
    true_labels,
    efficient_pred
)

In [ ]:
plot_confusion(
    true_labels,
    cnn_pred,
    "Custom CNN"
)

plot_confusion(
    true_labels,
    resnet_pred,
    "ResNet50"
)

plot_confusion(
    true_labels,
    efficient_pred,
    "EfficientNet"
)

In [ ]:
def plot_confusion(y_true, y_pred, title):

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(title)

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.show()

In [ ]:
import numpy as np

def dst_fusion_two(prob1, prob2):
    """
    Simple Dempster-Shafer combination for two probability vectors.
    """

    fused = (prob1 * prob2)

    fused = fused / np.sum(fused, axis=1, keepdims=True)

    prediction = np.argmax(fused, axis=1)

    return prediction, fused

In [ ]:
def dst_fusion_three(prob1, prob2, prob3):

    fused = prob1 * prob2 * prob3

    fused = fused / np.sum(fused, axis=1, keepdims=True)

    prediction = np.argmax(fused, axis=1)

    return prediction, fused

In [ ]:
cnn_resnet_pred, cnn_resnet_prob = dst_fusion_two(
    cnn_prob,
    resnet_prob
)

evaluate_model(
    "CustomCNN + ResNet50 (DST)",
    true_labels,
    cnn_resnet_pred
)

In [ ]:
cnn_eff_pred, cnn_eff_prob = dst_fusion_two(
    cnn_prob,
    efficient_prob
)

evaluate_model(
    "CustomCNN + EfficientNet (DST)",
    true_labels,
    cnn_eff_pred
)

In [ ]:
resnet_eff_pred, resnet_eff_prob = dst_fusion_two(
    resnet_prob,
    efficient_prob
)

evaluate_model(
    "ResNet50 + EfficientNet (DST)",
    true_labels,
    resnet_eff_pred
)

In [ ]:
results = pd.DataFrame({
    "Model / Combination": [
        "Custom CNN",
        "ResNet50",
        "EfficientNet-B0",
        "CustomCNN + ResNet50 (DST)",
        "CustomCNN + EfficientNet-B0 (DST)",
        "ResNet50 + EfficientNet-B0 (DST)",
        "CustomCNN + ResNet50 + EfficientNet-B0 (DST)"
    ],
    "Accuracy (%)": [
        accuracy_score(true_labels, cnn_pred) * 100,
        accuracy_score(true_labels, resnet_pred) * 100,
        accuracy_score(true_labels, efficient_pred) * 100,
        accuracy_score(true_labels, cnn_resnet_pred) * 100,
        accuracy_score(true_labels, cnn_eff_pred) * 100,
        accuracy_score(true_labels, resnet_eff_pred) * 100,
        accuracy_score(true_labels, all_pred) * 100
    ]
})

results = results.sort_values(
    by="Accuracy (%)",
    ascending=False
)

results

In [ ]:
# Three-model DST fusion
all_pred, all_prob = dst_fusion_three(
    cnn_prob,
    resnet_prob,
    efficient_prob
)

evaluate_model(
    "CustomCNN + ResNet50 + EfficientNet-B0 (DST)",
    true_labels,
    all_pred
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.axis('off')

# Convert numeric columns only
display_df = results.copy()
display_df["Accuracy (%)"] = display_df["Accuracy (%)"].map(lambda x: f"{x:.2f}")

table = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.3, 2)

plt.title(
    "FER2013 Emotion Recognition Results",
    fontsize=16,
    weight="bold",
    pad=20
)

plt.savefig(
    "/kaggle/working/dst_results_table.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

accuracy = [
    accuracy_score(true_labels, cnn_pred) * 100,
    accuracy_score(true_labels, resnet_pred) * 100,
    accuracy_score(true_labels, efficient_pred) * 100,
    accuracy_score(true_labels, cnn_resnet_pred) * 100,
    accuracy_score(true_labels, cnn_eff_pred) * 100,
    accuracy_score(true_labels, resnet_eff_pred) * 100,
    accuracy_score(true_labels, all_pred) * 100
]

models = [
    "Custom CNN",
    "ResNet50",
    "EfficientNet",
    "CNN+\nResNet",
    "CNN+\nEfficientNet",
    "ResNet+\nEfficientNet",
    "CNN+\nResNet+\nEfficientNet"
]

plt.figure(figsize=(12,6))

bars = plt.bar(models, accuracy)

plt.title("FER2013 Model Comparison using DST Fusion", fontsize=16)
plt.xlabel("Models")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)

# Show values on bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.5,
        f"{height:.2f}%",
        ha='center',
        fontsize=10
    )

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

plt.savefig(
    "/kaggle/working/model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

image = transform(img).unsqueeze(0).to(device)

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -----------------------------
# Emotion Classes
# -----------------------------
class_names = [
    "Angry",
    "Disgust",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise"
]

# -----------------------------
# Image Transform
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

# -----------------------------
# Load Image
# -----------------------------
image_path = "/kaggle/input/datasets/binu123456/my-dataset/happy.jpeg"

img = Image.open(image_path).convert("RGB")

plt.figure(figsize=(5,5))
plt.imshow(img)
plt.title("Input Image")
plt.axis("off")
plt.show()

image = transform(img).unsqueeze(0).to(device)

# -----------------------------
# Prediction Function
# -----------------------------
def predict(model):

    model.eval()

    with torch.no_grad():

        output = model(image)

        prob = F.softmax(output, dim=1).cpu().numpy()[0]

        pred = np.argmax(prob)

    return pred, prob

# -----------------------------
# Individual Predictions
# -----------------------------
cnn_label, cnn_prob = predict(customcnn)

resnet_label, resnet_prob = predict(resnet_model)

efficient_label, efficient_prob = predict(efficientnet_model)

# -----------------------------
# Fusion Function
# -----------------------------
def fuse(prob1, prob2):

    fused = prob1 * prob2

    fused = fused / fused.sum()

    pred = np.argmax(fused)

    return pred, fused

# -----------------------------
# Pairwise Fusion
# -----------------------------
cnn_resnet_label, cnn_resnet_prob = fuse(
    cnn_prob,
    resnet_prob
)

cnn_eff_label, cnn_eff_prob = fuse(
    cnn_prob,
    efficient_prob
)

resnet_eff_label, resnet_eff_prob = fuse(
    resnet_prob,
    efficient_prob
)

# -----------------------------
# Three Model Fusion
# -----------------------------
all_prob = cnn_prob * resnet_prob * efficient_prob

all_prob = all_prob / all_prob.sum()

all_label = np.argmax(all_prob)

# -----------------------------
# Print Predictions
# -----------------------------
print("="*70)

print(f"Custom CNN                      : {class_names[cnn_label]}")
print(f"ResNet50                        : {class_names[resnet_label]}")
print(f"EfficientNet-B0                 : {class_names[efficient_label]}")

print("-"*70)

print(f"Custom CNN + ResNet50           : {class_names[cnn_resnet_label]}")
print(f"Custom CNN + EfficientNet-B0    : {class_names[cnn_eff_label]}")
print(f"ResNet50 + EfficientNet-B0      : {class_names[resnet_eff_label]}")

print("-"*70)

print(f"All Models Fusion               : {class_names[all_label]}")

print("="*70)

# -----------------------------
# Probability Table
# -----------------------------
probability_table = pd.DataFrame({
    "Emotion": class_names,
    "Custom CNN": np.round(cnn_prob,4),
    "ResNet50": np.round(resnet_prob,4),
    "EfficientNet": np.round(efficient_prob,4),
    "CNN+ResNet": np.round(cnn_resnet_prob,4),
    "CNN+EfficientNet": np.round(cnn_eff_prob,4),
    "ResNet+EfficientNet": np.round(resnet_eff_prob,4),
    "All Models": np.round(all_prob,4)
})

display(probability_table)